# Lab 3 — NYC Yellow Taxi (Jan 2021): Profiling and Cleaning
INF4058 Data Engineering

Flow of this notebook: **Look → Profile (Task 1) → Find problems → Decide → Clean (Task 2) → Verify.**


## Step 1 — Look (first 100 rows)

In [1]:
import pandas as pd

file_path = "yellow_tripdata_2021-01.csv.gz"
chunk_size = 100_000

In [2]:
df = pd.read_csv(file_path, nrows=100)

print(df.head())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [3]:
print("Data types:")
print(df.dtypes)

print("\nMissing values in the first 100 rows:")
print(df.isna().sum())


Data types:
VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object

Missing values in the first 100 rows:
VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax          

In [4]:
print("\nShape of sample:")
print(df.shape)


Shape of sample:
(100, 18)


**What I see:** one row = one yellow taxi trip. `tpep_pickup_datetime` and `tpep_dropoff_datetime` are read as
strings (not real dates yet). In these 100 rows there are no nulls, but row 3 has `passenger_count = 0`, which looks odd.

## Step 2 — Profile the WHOLE file in chunks (Task 1)
Never load the whole file. The data is not modified.

In [5]:
total_rows = 0
null_counts = None
min_values = {}
max_values = {}
dtypes = {}

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):

    total_rows += len(chunk)

    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()

    for col in chunk.columns:
        dtypes[col] = str(chunk[col].dtype)

        col_min = chunk[col].min()
        col_max = chunk[col].max()

        if col not in min_values:
            min_values[col] = col_min
            max_values[col] = col_max
        else:
            if pd.notna(col_min):
                if pd.isna(min_values[col]) or col_min < min_values[col]:
                    min_values[col] = col_min
            if pd.notna(col_max):
                if pd.isna(max_values[col]) or col_max > max_values[col]:
                    max_values[col] = col_max

print("Total rows:", total_rows)
print("Matches known number 1,369,765:", total_rows == 1_369_765)

Total rows: 1369765
Matches known number 1,369,765: True


In [6]:
profile = pd.DataFrame({
    "dtype": pd.Series(dtypes),
    "null_count": null_counts,
    "null_percent": (null_counts / total_rows) * 100,
    "min": pd.Series(min_values),
    "max": pd.Series(max_values),
})

print(profile.to_string())

                         dtype  null_count  null_percent                  min                  max
VendorID               float64       98352       7.18021                    1                    2
tpep_pickup_datetime       str           0       0.00000  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime      str           0       0.00000  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count        float64       98352       7.18021                    0                    8
trip_distance          float64           0       0.00000                  0.0            263163.28
RatecodeID             float64       98352       7.18021                    1                   99
store_and_fwd_flag     float64       98352       7.18021                    N                    Y
PULocationID             int64           0       0.00000                    1                  265
DOLocationID             int64           0       0.00000                    1                  265
payment_ty

## Step 3 — Find problems (counts over the whole file)

In [7]:
null_passengers = 0
bad_dates = 0
bad_fares = 0
bad_distances = 0
zero_passengers = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):

    pickup = pd.to_datetime(chunk["tpep_pickup_datetime"], errors="coerce")

    null_passengers += chunk["passenger_count"].isna().sum()

    bad_dates += (
        (pickup < "2021-01-01") | (pickup >= "2021-02-01") | pickup.isna()
    ).sum()

    bad_fares += (chunk["fare_amount"] <= 0).sum()
    bad_distances += (chunk["trip_distance"] <= 0).sum()
    zero_passengers += (chunk["passenger_count"] == 0).sum()

In [8]:
problems = pd.DataFrame({
    "Problem": [
        "Null passenger_count",
        "Pickup outside January 2021",
        "Fare <= 0",
        "Trip distance <= 0",
        "Passenger count = 0",
    ],
    "Count": [
        null_passengers,
        bad_dates,
        bad_fares,
        bad_distances,
        zero_passengers,
    ],
})
problems["Percent"] = (problems["Count"] / total_rows * 100).round(3)

print(problems.to_string(index=False))

                    Problem  Count  Percent
       Null passenger_count  98352    7.180
Pickup outside January 2021     24    0.002
                  Fare <= 0   7411    0.541
         Trip distance <= 0  19952    1.457
        Passenger count = 0  26726    1.951


In [9]:
# Duplicate rows over the whole file (row hashes, chunk by chunk, so memory stays small)
seen = set()
duplicate_count = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
    hashes = pd.util.hash_pandas_object(chunk, index=False)
    for h in hashes:
        if h in seen:
            duplicate_count += 1
        else:
            seen.add(h)

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [10]:
# Spot-check one finding on a small sample, as the task suggests
sample = pd.read_csv(file_path, nrows=100_000)
print("Spot check, fare_amount <= 0 in first 100,000 rows:",
      len(sample.query("fare_amount <= 0")))
del sample

Spot check, fare_amount <= 0 in first 100,000 rows: 721


## Step 4 — Decide (drop, fix or flag — and who owns the rule)

| Problem | Decision | Owner | Why |
|---|---|---|---|
| null passenger_count | DROP | Data Engineering | cannot be trusted or analysed per passenger |
| pickup outside Jan 2021 | DROP | Data Engineering | the January file must only contain January trips |
| fare_amount <= 0 | DROP | Business / Finance | not a valid paid trip |
| trip_distance <= 0 | FLAG | Business | may be a cancelled trip, keep for audit |
| passenger_count = 0 | FLAG | Business | may be a recording error |
| duplicates | none needed | Data Engineering | count was 0 |

## Step 5 — Clean (Task 2)
`clean(df)` returns the cleaned df AND `{rule: rows_removed}`. It never changes the original.

In [11]:
def clean(df):
    cleaned = df.copy()          
    rows_removed = {}

    # Rule 1: DROP null passenger_count
    bad = cleaned["passenger_count"].isna()
    rows_removed["null_passenger_count"] = int(bad.sum())
    cleaned = cleaned.loc[~bad]

    # Rule 2: DROP pickup outside January 2021
    pickup = pd.to_datetime(cleaned["tpep_pickup_datetime"], errors="coerce")
    valid_date = (pickup >= "2021-01-01") & (pickup < "2021-02-01")
    rows_removed["pickup_outside_jan_2021"] = int((~valid_date).sum())
    cleaned = cleaned.loc[valid_date]

    # Rule 3: DROP fare_amount <= 0
    bad = cleaned["fare_amount"] <= 0
    rows_removed["fare_amount_le_0"] = int(bad.sum())
    cleaned = cleaned.loc[~bad]

    # Rules 4 and 5: FLAG only (nothing removed)
    cleaned = cleaned.copy()
    cleaned["flag_bad_distance"] = cleaned["trip_distance"] <= 0
    cleaned["flag_zero_passengers"] = cleaned["passenger_count"] == 0

    return cleaned, rows_removed

In [12]:
# Quick test on the 100-row sample (also proves the original is not changed)
df_before = df.copy()
sample_cleaned, sample_removed = clean(df)

print("Sample rows before:", len(df))
print("Sample rows after:", len(sample_cleaned))
print(sample_removed)
print("Original unchanged:", df.equals(df_before))

Sample rows before: 100
Sample rows after: 99
{'null_passenger_count': 0, 'pickup_outside_jan_2021': 0, 'fare_amount_le_0': 1}
Original unchanged: True


## Step 6 — Run clean() on the WHOLE file, chunk by chunk

In [13]:
rows_before = 0
rows_after = 0
totals = {}
flag_distance_total = 0
flag_passengers_total = 0

invalid_dates_left = 0
null_passengers_left = 0
fares_left = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):

    cleaned_chunk, removed_chunk = clean(chunk)

    rows_before += len(chunk)
    rows_after += len(cleaned_chunk)

    for rule, n in removed_chunk.items():
        totals[rule] = totals.get(rule, 0) + n

    flag_distance_total += int(cleaned_chunk["flag_bad_distance"].sum())
    flag_passengers_total += int(cleaned_chunk["flag_zero_passengers"].sum())

    # verification on the cleaned chunk
    p = pd.to_datetime(cleaned_chunk["tpep_pickup_datetime"], errors="coerce")
    invalid_dates_left += int(((p < "2021-01-01") | (p >= "2021-02-01") | p.isna()).sum())
    null_passengers_left += int(cleaned_chunk["passenger_count"].isna().sum())
    fares_left += int((cleaned_chunk["fare_amount"] <= 0).sum())

In [14]:
print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Rows removed:", rows_before - rows_after)
print()
print("Rows removed per rule:")
print(pd.DataFrame(list(totals.items()), columns=["Rule", "Rows Removed"]).to_string(index=False))
print()
print("Flagged (kept) rows: trip_distance <= 0 ->", flag_distance_total,
      "| passenger_count = 0 ->", flag_passengers_total)

Rows before: 1369765
Rows after:  1264125
Rows removed: 105640

Rows removed per rule:
                   Rule  Rows Removed
   null_passenger_count         98352
pickup_outside_jan_2021            24
       fare_amount_le_0          7264

Flagged (kept) rows: trip_distance <= 0 -> 17775 | passenger_count = 0 -> 26709


## Step 7 — Verify

In [15]:
print("1. rows before - rows after == sum(rows_removed):",
      rows_before - rows_after == sum(totals.values()))
print("2. rows before == 1,369,765:", rows_before == 1_369_765)
print("3. invalid pickup dates remaining:", invalid_dates_left)
print("4. null passenger_count remaining:", null_passengers_left)
print("5. fare_amount <= 0 remaining:", fares_left)
print("6. original sample unchanged:", df.equals(df_before))

1. rows before - rows after == sum(rows_removed): True
2. rows before == 1,369,765: True
3. invalid pickup dates remaining: 0
4. null passenger_count remaining: 0
5. fare_amount <= 0 remaining: 0
6. original sample unchanged: True
